# Chapter 13 Quiz 1~5

In [ ]:
import json
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd

BASE = "https://raw.githubusercontent.com/wesm/pydata-book/3rd-edition/datasets/"
DATA = Path("data")


def fetch(rel):
    f = DATA / rel
    if not f.exists():
        f.parent.mkdir(parents=True, exist_ok=True)
        tmp = f.with_name(f.name + ".part")
        urllib.request.urlretrieve(BASE + rel, tmp)
        tmp.replace(f)
    return f

## Quiz 1. Bitly 1.USA.gov

In [ ]:
with open(fetch("bitly_usagov/example.txt"), encoding="utf-8") as f:
    frame = pd.DataFrame([json.loads(line) for line in f])

frame = frame.dropna(subset=["a"]).copy()
frame["tz"] = frame["tz"].fillna("Missing").replace("", "Unknown")
frame["os"] = np.where(frame["a"].str.contains("Windows"), "Windows", "Not Windows")

top10 = frame["tz"].value_counts().nlargest(10).index
result = (
    frame[frame["tz"].isin(top10)]
    .groupby(["tz", "os"])
    .size()
    .reset_index(name="count")
)
result["ratio"] = result["count"] / result.groupby("tz")["count"].transform("sum")

display(result)
display(
    result[result["os"] == "Windows"]
    .sort_values("ratio", ascending=False)
    .head(5)
    .reset_index(drop=True)
)

,tz,os,count,ratio
0,America/Chicago,Not Windows,115,0.287500
1,America/Chicago,Windows,285,0.712500
2,America/Denver,Not Windows,132,0.691099
3,America/Denver,Windows,59,0.308901
4,America/Los_Angeles,Not Windows,130,0.340314
5,America/Los_Angeles,Windows,252,0.659686
6,America/New_York,Not Windows,339,0.270983
7,America/New_York,Windows,912,0.729017
8,America/Sao_Paulo,Not Windows,13,0.393939
9,America/Sao_Paulo,Windows,20,0.606061


,tz,os,count,ratio
0,Pacific/Honolulu,Windows,36,1.000000
1,Asia/Tokyo,Windows,35,0.945946
2,America/New_York,Windows,912,0.729017
3,America/Chicago,Windows,285,0.712500
4,America/Los_Angeles,Windows,252,0.659686


**답:** 접속 건수 상위 10개 시간대 중 Windows 사용자 비율이 가장 높은 5곳은 Pacific/Honolulu(100%, 36건), Asia/Tokyo(94.6%), America/New_York(72.9%), America/Chicago(71.3%), America/Los_Angeles(66.0%)이다.

Honolulu와 Tokyo는 접속 건수가 각각 36건, 37건으로 적어 비율이 극단적으로 나왔고, 접속이 가장 많은 New_York(1,251건)에서도 약 73%가 Windows 사용자이다.

## Quiz 2. MovieLens 1M

In [ ]:
def read_dat(name, cols):
    return pd.read_csv(
        fetch(f"movielens/{name}.dat"),
        sep="::", header=None, names=cols, engine="python", encoding="latin-1",
    )


users = read_dat("users", ["user_id", "gender", "age", "occupation", "zip"])
ratings = read_dat("ratings", ["user_id", "movie_id", "rating", "timestamp"])
movies = read_dat("movies", ["movie_id", "title", "genres"])

data = ratings.merge(users, on="user_id").merge(movies, on="movie_id")

title_counts = data.groupby("title").size()
data = data[data["title"].isin(title_counts[title_counts >= 250].index)]

mean_ratings = data.pivot_table(
    values="rating", index="title", columns="gender", aggfunc="mean"
)
mean_ratings["diff"] = mean_ratings["M"] - mean_ratings["F"]

movie_genres = (
    movies.assign(genre=movies["genres"].str.split("|"))
    .explode("genre")[["title", "genre"]]
    .merge(mean_ratings["diff"].reset_index(), on="title")
)

idx = movie_genres["diff"].abs().groupby(movie_genres["genre"]).idxmax()
result = (
    movie_genres.loc[idx, ["genre", "title", "diff"]]
    .sort_values("diff")
    .reset_index(drop=True)
)
display(result)

,genre,title,diff
0,Romance,Dirty Dancing (1987),-0.830782
1,Musical,Dirty Dancing (1987),-0.830782
2,Thriller,Jumpin' Jack Flash (1986),-0.676359
3,Children's,Anastasia (1997),-0.518391
4,Animation,Anastasia (1997),-0.518391
5,Mystery,Sommersby (1993),-0.435603
6,Fantasy,Labyrinth (1986),-0.392001
7,Documentary,"Thin Blue Line, The (1988)",-0.183282
8,Film-Noir,Touch of Evil (1958),0.343935
9,Crime,From Dusk Till Dawn (1996),0.496844


**답:** 평점이 250개 이상인 영화를 장르별로 펼친 뒤, 장르마다 남녀 평점 차이(diff = 남성 평균 − 여성 평균)의 절댓값이 가장 큰 영화 1편을 골랐다. diff가 음수이면 여성이, 양수이면 남성이 더 높게 평가한 영화이다.

여성이 더 높게 평가한 차이가 가장 큰 영화는 Romance·Musical 장르의 Dirty Dancing(−0.83)이고, 남성이 더 높게 평가한 차이가 가장 큰 영화는 Action·Western 장르의 The Good, The Bad and The Ugly(+0.73)이다. 한 영화가 여러 장르에 속하면 각 장르에 중복해서 나타난다.

## Quiz 3. US Baby Names 1880-2010

In [ ]:
def read_year(year):
    df = pd.read_csv(fetch(f"babynames/yob{year}.txt"), names=["name", "sex", "births"])
    df["year"] = year
    return df


with ThreadPoolExecutor(8) as pool:
    names = pd.concat(pool.map(read_year, range(1880, 2011)), ignore_index=True)

names["decade"] = names["year"] // 10 * 10

table = (
    names.groupby(["name", "decade", "sex"])["births"]
    .sum()
    .unstack("sex", fill_value=0)
)
table["total"] = table["F"] + table["M"]
table["f_ratio"] = table["F"] / table["total"]

wide = table[["total", "f_ratio"]].unstack("decade")

mask = (
    (wide[("total", 1900)] >= 500)
    & (wide[("total", 2000)] >= 500)
    & (wide[("f_ratio", 1900)] < 0.2)
    & (wide[("f_ratio", 2000)] > 0.8)
)
result = wide.loc[mask, [("f_ratio", 1900), ("f_ratio", 2000)]]
result.columns = ["f_ratio_1900s", "f_ratio_2000s"]
display(result.sort_values("f_ratio_1900s"))

,f_ratio_1900s,f_ratio_2000s
name,,
Aubrey,0.104668,0.951207
Leslie,0.113485,0.969772


**답:** 1900년대에는 여아 비율이 20% 미만이었다가 2000년대에는 80%를 넘은, 즉 남자 이름에서 여자 이름으로 바뀐 이름은 **Aubrey**(10.5% → 95.1%)와 **Leslie**(11.3% → 97.0%) 두 개이다.

## Quiz 4. USDA Food Database

In [ ]:
with open(fetch("usda_food/database.json"), encoding="utf-8") as f:
    db = json.load(f)

info = pd.DataFrame(db, columns=["description", "group", "id", "manufacturer"])
info = info.rename(columns={"description": "food", "group": "fgroup"})

nutrients = pd.concat(
    [pd.DataFrame(rec["nutrients"]).assign(id=rec["id"]) for rec in db],
    ignore_index=True,
)
nutrients = nutrients.drop_duplicates().rename(
    columns={"description": "nutrient", "group": "nutgroup"}
)

ndata = nutrients.merge(info, on="id")

protein = ndata[ndata["nutrient"] == "Protein"].copy()
protein["fg_median"] = protein.groupby("fgroup")["value"].transform("median")
protein["ratio"] = protein["value"] / protein["fg_median"]

high = protein[(protein["fg_median"] > 0) & (protein["ratio"] >= 2)]

summary = high.groupby("fgroup").agg(
    count=("id", "size"), max_ratio=("ratio", "max")
)
best = high.loc[
    high.groupby("fgroup")["ratio"].idxmax(), ["fgroup", "food", "value"]
].set_index("fgroup")

result = summary.join(best).sort_values("count", ascending=False)
display(result)

,count,max_ratio,food,value
fgroup,,,,
Vegetables and Vegetable Products,139,29.025253,"Seaweed, spirulina, dried",57.47
Beverages,106,358.571429,"Cocoa mix, with aspartame, low calorie, powder...",25.10
Legumes and Legume Products,101,8.316384,"Soy protein isolate, potassium type, crude pro...",88.32
Sweets,69,27.612903,"Gelatins, dry powder, unsweetened",85.60
Fruits and Fruit Juices,64,8.099174,"Apricots, dehydrated (low-moisture), sulfured,...",4.90
"Soups, Sauces, and Gravies",57,14.767442,"CAMPBELL Soup Company, CAMPBELL'S Red and Whit...",25.40
Baby Foods,53,6.962025,"Child formula, MEAD JOHNSON, PORTAGEN, with ir...",16.50
Fats and Oils,34,124.562500,"Salad dressing, russian dressing, low calorie",19.93
Ethnic Foods,31,7.059829,"Seal, bearded (Oogruk), meat, dried (Alaska Na...",82.60


**답:** 단백질 함량이 자기 식품군 중위수의 2배 이상인 식품이 가장 많은 식품군은 Vegetables and Vegetable Products(139개), Beverages(106개), Legumes and Legume Products(101개) 순이다.

식품군별 최대 ratio는 Beverages가 358.6배로 가장 큰데(Cocoa mix, 25.1g), 음료의 단백질 중위수가 약 0.07g으로 거의 0이어서 비율이 크게 커진 것이다. 그다음은 Fats and Oils 124.6배(Salad dressing, russian dressing, low calorie, 19.93g)이다. 표의 food와 value 열이 각 식품군에서 ratio가 가장 큰 식품의 이름과 단백질 함량이다.

## Quiz 5. 2012 FEC

In [ ]:
try:
    fec = pd.read_parquet(fetch("fec/fec.parquet"))
except Exception:
    fec = pd.read_csv(fetch("fec/P00000001-ALL.csv"), low_memory=False, index_col=False)

fec = fec[fec["contb_receipt_amt"] > 0].copy()
fec["cand"] = fec["cand_nm"].str.split(",").str[0]
fec = fec[fec["cand"].isin(["Obama", "Romney"])].copy()

fec["contbr_occupation"] = fec["contbr_occupation"].replace({
    "INFORMATION REQUESTED PER BEST EFFORTS": "NOT PROVIDED",
    "INFORMATION REQUESTED": "NOT PROVIDED",
    "INFORMATION REQUESTED (BEST EFFORTS)": "NOT PROVIDED",
    "C.E.O.": "CEO",
})

totals = (
    fec.groupby(["contbr_st", "cand"])["contb_receipt_amt"]
    .sum()
    .unstack(fill_value=0)
)
totals = totals[totals.sum(axis=1) > 100_000]
ratios = totals.div(totals.sum(axis=1), axis=0)

top_occupation = (
    fec.groupby(["contbr_st", "contbr_occupation"])["contb_receipt_amt"]
    .sum()
    .reset_index()
    .sort_values("contb_receipt_amt")
    .drop_duplicates("contbr_st", keep="last")
    .set_index("contbr_st")["contbr_occupation"]
)
ratios["top_occupation"] = top_occupation

display(ratios.sort_values("Obama", ascending=False).head(5))

cand,Obama,Romney,top_occupation
contbr_st,,,
VT,0.946983,0.053017,RETIRED
ME,0.908825,0.091175,RETIRED
PR,0.881192,0.118808,PRESIDENT
HI,0.876774,0.123226,RETIRED
MN,0.855913,0.144087,RETIRED


**답:** Obama 후원금 비율이 가장 높은 주 5곳은 VT(94.7%), ME(90.9%), PR(88.1%), HI(87.7%), MN(85.6%)이다. 이 지역들에서 후원금 합계가 가장 큰 직업은 PR(PRESIDENT)을 제외하면 모두 RETIRED(은퇴자)이다.